# Evaluate the trained classifier

Run this locally (CPU is fine — evaluation is one forward pass per test
image, not training) after downloading `models/best_model.pt` from the
Colab training notebook.

This is a go/no-go checkpoint (see project plan, Phase 6): review the
confusion matrix here before building the rest of the pipeline around
this specific trained model.

In [ ]:
from pathlib import Path

import torch

from hieroglyph.data.dataset import build_dataloaders
from hieroglyph.models.classifier import load_checkpoint

CHECKPOINT_PATH = Path("..") / "models" / "best_model.pt"
DATA_ROOT = Path("..") / "data" / "raw"

assert CHECKPOINT_PATH.exists(), (
    f"No checkpoint at {CHECKPOINT_PATH.resolve()} -- download best_model.pt "
    "from the Colab training notebook first."
)

model, class_to_idx = load_checkpoint(CHECKPOINT_PATH)
idx_to_class = {idx: name for name, idx in class_to_idx.items()}

# batch_size/split here only need to match what the checkpoint's
# class_to_idx was built from (same data_root) -- the split itself is
# deterministic (fixed seed in split_dataset), so this reproduces the same
# test set that was held out during training.
_, _, test_loader, loaded_class_to_idx = build_dataloaders(DATA_ROOT, batch_size=32)
assert loaded_class_to_idx == class_to_idx, "Class mapping mismatch between checkpoint and current data/raw"

## Overall test accuracy

In [ ]:
all_preds, all_labels = [], []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

accuracy = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f"test accuracy: {accuracy:.3f}  ({len(all_labels)} test images)")
print(
    "Remember: ~51 classes have no test representation at all (routed "
    "entirely to train in Phase 2/3) -- this accuracy number only reflects "
    "the ~120 classes that had enough images to hold some back."
)

## Per-class precision / recall / F1

In [ ]:
from sklearn.metrics import classification_report

present_labels = sorted(set(all_labels))
target_names = [idx_to_class[i] for i in present_labels]

print(
    classification_report(
        all_labels, all_preds, labels=present_labels, target_names=target_names, zero_division=0
    )
)

## Confusion matrix

Full 171x171 is unreadable as a plot -- instead, list the sign *pairs* the
model confuses most often. This is the part worth actually reading: which
signs looks alike to the model, and does that match what looks alike to a
human eye (a sanity check on whether errors are "reasonable" or bizarre).

In [ ]:
from collections import Counter

confusions = Counter(
    (idx_to_class[true], idx_to_class[pred])
    for true, pred in zip(all_labels, all_preds)
    if true != pred
)

print("Most common misclassifications (true_sign -> predicted_sign: count):")
for (true_sign, pred_sign), count in confusions.most_common(20):
    print(f"  {true_sign} -> {pred_sign}: {count}")

## Record the baseline

Copy the test accuracy number into the README once you've reviewed the
results above and are ready to treat this checkpoint as the one the rest
of the pipeline (segmentation, lookup, inference, Streamlit app) builds on.